# Preprocessing et Feature Engineering

In [6]:
import pandas as pd
import numpy as np

## Date conversion and Data loading

In [9]:
df = pd.read_csv("../data/merged_dataset.csv")
df["delivery_time"] = pd.to_datetime(df['delivery_time'])

## Handling Missing Values

Linear interpolation for weather, as it's a continuous time series

In [3]:
df = df.groupby('site_name').apply(lambda x: x.interpolate(method='linear', limit_direction='both')).reset_index(drop=True)

C:\Users\clemm\AppData\Local\Temp\ipykernel_73124\19154904.py:1: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df = df.groupby('site_name').apply(lambda x: x.interpolate(method='linear', limit_direction='both')).reset_index(drop=True)
C:\Users\clemm\AppData\Local\Temp\ipykernel_73124\19154904.py:1: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df = df.groupby('site_name').apply(lambda x: x.interpolate(method='linear', limit_direction='both')).reset_index(drop=True)
C:\Users\clemm\AppData\Local\Temp\ipykernel_73124\19154904.py:1: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df = df.groupby('site_name').apply(lambda

Power is proportional to the cube of wind speed:

In [4]:
df['wind_speed_100m_cube'] = df['wind_speed_100m'] ** 3

## Feature Engineering: Wind Physics

Wind Shear: Difference in speed between 10m and 100m:

In [5]:
df['wind_shear'] = df['wind_speed_100m'] - df['wind_speed_10m']

## Feature Engineering: Circular Encoding (Direction & Time)

Wind Direction (avoiding the 359° to 0° jump)

In [7]:
df['wind_dir_100m_sin'] = np.sin(np.radians(df['wind_direction_100m']))
df['wind_dir_100m_cos'] = np.cos(np.radians(df['wind_direction_100m']))

Time-based features (Daily and Seasonal cycles)

In [10]:
df['hour'] = df['delivery_time'].dt.hour
df['month'] = df['delivery_time'].dt.month

df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 23)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 23)
df['month_sin'] = np.sin(2 * np.pi * (df['month'] - 1) / 11)
df['month_cos'] = np.cos(2 * np.pi * (df['month'] - 1) / 11)

## Target Normalization (Critical for multi-site modeling)
Predicting Load Factor (0 to 1) instead of raw MW

In [11]:
df['target'] = df['production'] / df['installed_capacity']

Remove columns that would cause data leakage or are redundant

In [13]:
cols_to_drop = ['production', 'hour', 'month', 'wind_direction_10m', 'wind_direction_100m']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])